# 🧪 Spike LangGraph - MVP Analyse Verbatims

**Objectif**: Valider l'utilisation de LangGraph pour l'orchestration du mode Full Coverage.

**Points à valider**:
1. Création d'un graph simple avec 2 nodes
2. Parallélisation des appels LLM
3. Gestion des erreurs et retry
4. Agrégation des résultats

---

## 1. Setup

In [ ]:
# Installation (si nécessaire)
# !pip install langgraph langchain openai python-dotenv

In [ ]:
import os
import sys
from pathlib import Path

# Ajouter le répertoire parent au path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Charger les variables d'environnement
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print(f"Project root: {project_root}")
print(f"OpenAI API Key configurée: {'Oui' if os.getenv('OPENAI_API_KEY') else 'Non'}")

In [ ]:
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json
from loguru import logger

## 2. Définition du State

In [ ]:
class ChunkResult(TypedDict):
    """Résultat d'analyse d'un chunk."""
    chunk_index: int
    topics: List[dict]
    status: str  # 'success' | 'failed'
    error: str | None


class AnalysisState(TypedDict):
    """État global du graph d'analyse."""
    # Input
    brief: str
    chunks: List[List[str]]  # Liste de chunks (chaque chunk = liste de verbatims)
    
    # Processing
    chunk_results: Annotated[List[ChunkResult], operator.add]  # Agrégation automatique
    current_chunk_index: int
    
    # Output
    merged_topics: List[dict]
    status: str

## 3. Définition des Nodes

In [ ]:
# Client LLM
llm = ChatOpenAI(
    model="gpt-4-turbo-preview",
    temperature=0.7,
    timeout=120,
)


def analyze_chunk(state: AnalysisState, chunk_index: int) -> ChunkResult:
    """
    Analyse un chunk de verbatims.
    """
    chunk = state["chunks"][chunk_index]
    brief = state["brief"]
    
    logger.info(f"Analyzing chunk {chunk_index} ({len(chunk)} verbatims)")
    
    # Construire le prompt
    verbatims_text = "\n".join(f"[{i+1}] {v}" for i, v in enumerate(chunk))
    
    prompt = f"""
    Contexte: {brief}
    
    Verbatims:
    {verbatims_text}
    
    Identifie les thèmes principaux. Réponds en JSON:
    {{
      "themes": [
        {{"topic_label": "...", "subtopic_label": "...", "count": N}}
      ]
    }}
    """
    
    try:
        response = llm.invoke([
            SystemMessage(content="Tu es un expert en analyse de verbatims. Réponds uniquement en JSON valide."),
            HumanMessage(content=prompt)
        ])
        
        # Parser la réponse
        content = response.content
        # Nettoyer le markdown si présent
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]
            
        data = json.loads(content.strip())
        
        return ChunkResult(
            chunk_index=chunk_index,
            topics=data.get("themes", []),
            status="success",
            error=None
        )
        
    except Exception as e:
        logger.error(f"Error analyzing chunk {chunk_index}: {e}")
        return ChunkResult(
            chunk_index=chunk_index,
            topics=[],
            status="failed",
            error=str(e)
        )

In [ ]:
def process_chunks_node(state: AnalysisState) -> dict:
    """
    Node: Traite tous les chunks (séquentiellement pour ce spike).
    """
    results = []
    
    for i in range(len(state["chunks"])):
        result = analyze_chunk(state, i)
        results.append(result)
    
    return {"chunk_results": results}


def merge_topics_node(state: AnalysisState) -> dict:
    """
    Node: Fusionne les thèmes de tous les chunks.
    """
    logger.info("Merging topics from all chunks")
    
    # Collecter tous les thèmes
    all_topics = []
    for result in state["chunk_results"]:
        if result["status"] == "success":
            all_topics.extend(result["topics"])
    
    # Fusion simple: regrouper par label (Pass 1 simplifié)
    topic_counts = {}
    for t in all_topics:
        label = t.get("topic_label", "Unknown")
        if label not in topic_counts:
            topic_counts[label] = {
                "topic_label": label,
                "subtopic_label": t.get("subtopic_label"),
                "count": 0,
                "sources": []
            }
        topic_counts[label]["count"] += t.get("count", 1)
    
    merged = list(topic_counts.values())
    merged.sort(key=lambda x: x["count"], reverse=True)
    
    # Déterminer le statut global
    failed_chunks = sum(1 for r in state["chunk_results"] if r["status"] == "failed")
    total_chunks = len(state["chunk_results"])
    
    if failed_chunks == 0:
        status = "success"
    elif failed_chunks < total_chunks:
        status = "partial"
    else:
        status = "failed"
    
    logger.info(f"Merged {len(all_topics)} topics into {len(merged)} unique topics")
    
    return {
        "merged_topics": merged,
        "status": status
    }

## 4. Construction du Graph

In [ ]:
def build_analysis_graph():
    """
    Construit le graph LangGraph pour l'analyse.
    """
    # Créer le graph
    workflow = StateGraph(AnalysisState)
    
    # Ajouter les nodes
    workflow.add_node("process_chunks", process_chunks_node)
    workflow.add_node("merge_topics", merge_topics_node)
    
    # Définir les edges
    workflow.set_entry_point("process_chunks")
    workflow.add_edge("process_chunks", "merge_topics")
    workflow.add_edge("merge_topics", END)
    
    # Compiler
    return workflow.compile()


# Créer le graph
graph = build_analysis_graph()
print("✅ Graph créé avec succès")

In [ ]:
# Visualiser le graph (si mermaid disponible)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation non disponible: {e}")
    print("\nStructure du graph:")
    print("  [START] -> process_chunks -> merge_topics -> [END]")

## 5. Test avec données fictives

In [ ]:
# Données de test
test_verbatims = [
    # Chunk 1
    [
        "La livraison était super rapide, reçu en 2 jours!",
        "Produit de qualité mais emballage abîmé",
        "Très déçu par le délai de livraison, 2 semaines d'attente",
        "Le service client m'a bien aidé pour mon retour",
        "Prix un peu élevé mais qualité au rendez-vous",
    ],
    # Chunk 2
    [
        "Livraison express nickel!",
        "Le produit ne correspond pas à la description",
        "Bon rapport qualité prix",
        "SAV réactif et efficace",
        "Emballage soigné, rien à dire",
    ],
    # Chunk 3
    [
        "Délai de livraison trop long",
        "Excellente qualité, je recommande",
        "Service après-vente inexistant",
        "Trop cher pour ce que c'est",
        "Conforme à mes attentes",
    ]
]

print(f"Test avec {len(test_verbatims)} chunks, {sum(len(c) for c in test_verbatims)} verbatims au total")

In [ ]:
# Exécuter le graph
initial_state = {
    "brief": "Analyse des avis clients sur un site e-commerce. Identifier les thèmes principaux de satisfaction et insatisfaction.",
    "chunks": test_verbatims,
    "chunk_results": [],
    "current_chunk_index": 0,
    "merged_topics": [],
    "status": "pending"
}

print("🚀 Lancement de l'analyse...")
print("="*50)

result = graph.invoke(initial_state)

print("="*50)
print(f"\n✅ Analyse terminée avec statut: {result['status']}")

In [ ]:
# Afficher les résultats
print("\n📊 RÉSULTATS")
print("="*50)

print(f"\nStatut global: {result['status']}")
print(f"Chunks traités: {len(result['chunk_results'])}")

# Détail par chunk
print("\n📦 Détail par chunk:")
for cr in result['chunk_results']:
    status_emoji = "✅" if cr['status'] == 'success' else "❌"
    print(f"  Chunk {cr['chunk_index']}: {status_emoji} {cr['status']} - {len(cr['topics'])} thèmes")

# Thèmes fusionnés
print("\n🏷️ Thèmes identifiés (après fusion):")
for i, topic in enumerate(result['merged_topics'][:10], 1):
    label = topic['topic_label']
    count = topic.get('count', 'N/A')
    print(f"  {i}. {label} (count: {count})")

## 6. Test avec parallélisation (asyncio)

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import time


async def process_chunks_parallel(state: AnalysisState, max_concurrent: int = 5) -> dict:
    """
    Version parallèle du traitement des chunks.
    """
    loop = asyncio.get_event_loop()
    
    with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
        # Créer les futures
        futures = [
            loop.run_in_executor(executor, analyze_chunk, state, i)
            for i in range(len(state["chunks"]))
        ]
        
        # Attendre tous les résultats
        results = await asyncio.gather(*futures)
    
    return {"chunk_results": list(results)}


# Test de performance
print("⏱️ Comparaison séquentiel vs parallèle")
print("="*50)

# Note: pour un vrai test, utilisez plus de chunks
# Ici c'est juste pour démontrer le pattern

## 7. Conclusions du Spike

### ✅ Ce qui fonctionne
- LangGraph permet de définir un workflow clair
- La gestion d'état avec TypedDict est propre
- L'agrégation automatique avec `Annotated[..., operator.add]` simplifie le code

### ⚠️ Points d'attention
- Parallélisation: LangGraph ne parallélise pas nativement, il faut utiliser asyncio/ThreadPool
- Retry: à implémenter manuellement ou via tenacity
- State size: attention à la taille du state si beaucoup de chunks

### 📋 Recommandations pour le MVP
1. **Utiliser LangGraph** pour la structure du workflow
2. **Implémenter la parallélisation** avec asyncio + semaphore pour limiter les appels
3. **Ajouter tenacity** pour les retries avec backoff
4. **Stocker les résultats intermédiaires** en DB (pas dans le state) pour les gros datasets

### 🚫 Alternative si LangGraph trop complexe
- Simple orchestration avec asyncio + classes Python
- Moins élégant mais plus simple à debugger

In [ ]:
# Sauvegarder les conclusions
conclusions = {
    "decision": "GO",  # ou "NO-GO" si problèmes
    "langgraph_version": "0.0.20",
    "points_positifs": [
        "Workflow structuré",
        "State management propre",
        "Extensible"
    ],
    "points_attention": [
        "Parallélisation manuelle",
        "Retry à implémenter"
    ],
    "recommandation": "Utiliser LangGraph avec wrapper asyncio pour parallélisation"
}

print("\n" + "="*50)
print("📋 DÉCISION FINALE")
print("="*50)
print(f"\n🎯 {conclusions['decision']}: {conclusions['recommandation']}")